# Public grievances in Odisha: how they arrive, and how long they take

Everything below covers **1.37 million grievances** filed between April 2021 and
July 2025, and the 6.6 million actions officers recorded against them.

Years run **July to June**. A year labelled *2024-25* means July 2024 through
June 2025. Two years at the edges of the data are incomplete and are marked
**(part year)** wherever they appear: *2020-21* holds only April-June 2021, and
*2025-26* only July 2025. Their totals are smaller because the window is
shorter, not because activity fell.

Each section opens with the same three things: the figure overall, the figure
year by year, and a month-by-month chart.

**What this is for.** The analysis exists to inform which districts we choose for
field deep dives. Sections 1 to 5 establish how the system behaves as a whole;
**section 6 is the district-by-district view that the choice should rest on.**

In [ ]:
import tempfile
from pathlib import Path

from IPython.display import Markdown, display

import matplotlib as mpl
import matplotlib.dates as mdates
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import polars as pl

from janasunani.olap.lake import connect
from janasunani.analytics.marts import install

TABLES = ["complaints", "action_history"]
con = connect(tables=TABLES)

# Bound the engine. Uncapped, DuckDB sizes its buffer pool from total system RAM
# and never spills; an earlier run of this notebook peaked at 55 GB.
con.execute("SET memory_limit = '8GB'")
con.execute(f"SET temp_directory = '{tempfile.gettempdir()}/duckdb_spill'")
con.execute("SET preserve_insertion_order = false")


def q(sql: str) -> pl.DataFrame:
    return con.execute(sql).pl()


# Years run July-June throughout.
con.execute("""
CREATE OR REPLACE MACRO fy_start(d) AS
  CASE WHEN EXTRACT(MONTH FROM d) >= 7
       THEN EXTRACT(YEAR FROM d) ELSE EXTRACT(YEAR FROM d) - 1 END
""")
con.execute("""
CREATE OR REPLACE MACRO fy_label(d) AS
  CAST(fy_start(d) AS INT) || '-' || RIGHT(CAST(fy_start(d) + 1 AS VARCHAR), 2)
""")

# The two years the data only partly covers.
PART_YEARS = {"2020-21", "2025-26"}


def year_col(df: pl.DataFrame, col: str = "year") -> pl.DataFrame:
    """Mark the part-covered years so no one reads a short window as a decline."""
    return df.with_columns(
        pl.when(pl.col(col).is_in(list(PART_YEARS)))
        .then(pl.col(col) + pl.lit(" (part year)"))
        .otherwise(pl.col(col))
        .alias(col)
    )

In [ ]:
# --- House chart style: DPIC brand -------------------------------------------
# Palette is the DPIC design system's (dpic.branding.colors). Validated with the
# dataviz palette checker: maroon/blue/orange clear both discrimination gates
# all-pairs (CVD dE 14.8, normal-vision dE 20.8). A 4th categorical hue fails the
# normal-vision floor, so three series is a hard cap -- past that, use one hue
# and sort by value instead of adding colours.
MAROON, BLUE, ORANGE = "#8B1524", "#155F83", "#C16622"
SERIES = [MAROON, BLUE, ORANGE]
INK, INK_SOFT, RULE = "#1A1A1A", "#666666", "#DDDDDD"

# Calibri is the DPIC brand face. Resolve it once against what is actually
# installed: leaving a missing family in the list makes matplotlib emit a
# findfont warning for every text object, which buried the real output.
_installed = {f.name for f in mpl.font_manager.fontManager.ttflist}
FONT = next((f for f in ("Calibri", "Helvetica Neue", "Arial", "DejaVu Sans")
             if f in _installed), "DejaVu Sans")

mpl.rcParams.update({
    "figure.dpi": 130,
    "savefig.dpi": 130,
    "figure.facecolor": "white",
    "axes.facecolor": "white",
    "font.family": FONT,
    "font.size": 10,
    "axes.titlesize": 12,
    "axes.titleweight": "bold",
    "axes.titlecolor": INK,
    "axes.titlelocation": "left",
    "axes.labelcolor": INK_SOFT,
    "axes.labelsize": 9.5,
    "axes.edgecolor": RULE,
    "axes.spines.top": False,
    "axes.spines.right": False,
    "axes.grid": True,
    "grid.color": RULE,
    "grid.linewidth": 0.6,
    "xtick.color": INK_SOFT,
    "ytick.color": INK_SOFT,
    "xtick.labelsize": 9,
    "ytick.labelsize": 9,
    "legend.frameon": False,
    "legend.fontsize": 9,
    "lines.linewidth": 2.0,
    "lines.markersize": 5,
})

FIGDIR = Path("../outputs/figures")
FIGDIR.mkdir(parents=True, exist_ok=True)


def finish(ax, title, subtitle=None, note=None, xgrid=False, ygrid=True, save=None):
    """Titles, a recessive grid, and the caveat line that must travel with a chart."""
    if subtitle:
        ax.set_title(title, pad=32)
        ax.text(0, 1.022, subtitle, transform=ax.transAxes,
                fontsize=9.5, color=INK_SOFT, va="bottom")
    else:
        ax.set_title(title, pad=12)
    ax.xaxis.grid(xgrid)
    ax.yaxis.grid(ygrid)
    ax.set_axisbelow(True)
    if note:
        ax.figure.text(0.005, -0.02, note, fontsize=8, color=INK_SOFT, va="top")
    ax.figure.tight_layout()
    if save:
        ax.figure.savefig(FIGDIR / f"{save}.png", bbox_inches="tight", facecolor="white")
    plt.show()


def label_points(ax, xs, ys, labels, fontsize=7.8):
    """Direct labels that step out of each other's way.

    Tries a few offsets per point and takes the first that does not overlap a
    label already placed. Beats hand-tuning when two districts sit on the same
    spot, which several do.
    """
    ax.figure.canvas.draw()
    placed = []
    cands = [(5, 3), (5, -10), (-5, 3), (-5, -10), (5, 12), (5, -20)]
    for x, y, text in sorted(zip(xs, ys, labels), key=lambda t: -t[0]):
        px, py = ax.transData.transform((x, y))
        w, h = len(text) * fontsize * 0.55, fontsize * 1.35
        for dx, dy in cands:
            ha = "left" if dx > 0 else "right"
            x0 = px + dx if dx > 0 else px + dx - w
            box = (x0, py + dy - h / 2, x0 + w, py + dy + h / 2)
            if not any(box[0] < b[2] and b[0] < box[2] and
                       box[1] < b[3] and b[1] < box[3] for b in placed):
                placed.append(box)
                ax.annotate(text, (x, y), xytext=(dx, dy), textcoords="offset points",
                            fontsize=fontsize, color=INK, ha=ha, va="center")
                break


def thousands(ax, axis="y"):
    fmt = mticker.FuncFormatter(lambda v, _: f"{v:,.0f}")
    (ax.yaxis if axis == "y" else ax.xaxis).set_major_formatter(fmt)


def barh(ax, labels, values, color=MAROON, fmt="{:,.0f}"):
    """Horizontal bars, sorted by the caller, with a direct label on each end."""
    # DuckDB hands back DECIMAL for ratio columns, which will not multiply with
    # the float padding below.
    values = [float(v) for v in values]
    y = range(len(labels))
    ax.barh(list(y), list(values), color=color, height=0.68, zorder=3)
    ax.set_yticks(list(y), labels)
    ax.invert_yaxis()
    span = max(values) if len(values) else 1
    for i, v in enumerate(values):
        ax.text(v + span * 0.012, i, fmt.format(v), va="center",
                fontsize=9, color=INK)
    ax.set_xlim(0, span * 1.16)
    ax.spines["left"].set_color(RULE)
    return ax

In [ ]:
def facts(title, sql_from, date_col, metric="COUNT(*)", metric_name="Grievances",
          where="TRUE", monthly_title=None, monthly_ylabel=None, save=None,
          fmt="{:,.0f}", chart=True):
    """The three openers every section gets: overall, by year, and by month.

    `metric` is any aggregate expression over `sql_from`, so this serves counts
    and medians alike.
    """
    overall = q(f"SELECT {metric} AS v FROM {sql_from} WHERE {where}")["v"][0]
    display(Markdown(f"**{title}: {fmt.format(overall)}**"))

    by_year = year_col(q(f"""
        SELECT fy_label({date_col}) AS year, {metric} AS value
        FROM {sql_from} WHERE {where}
        GROUP BY 1 ORDER BY 1
    """))
    display(by_year.rename({"value": metric_name}))

    if not chart:
        return by_year

    monthly = q(f"""
        SELECT DATE_TRUNC('month', {date_col}) AS month, {metric} AS value
        FROM {sql_from} WHERE {where}
        GROUP BY 1 ORDER BY 1
    """)
    fig, ax = plt.subplots(figsize=(9, 3.3))
    ax.plot(monthly["month"], monthly["value"], color=MAROON, zorder=3)
    ax.fill_between(monthly["month"], monthly["value"], color=MAROON, alpha=0.10, zorder=2)
    ax.set_ylim(bottom=0)
    ax.set_ylabel(monthly_ylabel or metric_name)
    thousands(ax)
    finish(ax, monthly_title or f"{title}, by month",
           subtitle="Years run July to June. First and last months are partial.",
           save=save)
    return by_year

In [ ]:
# Two working tables. `channel` is our own grouping of the 15 filing modes into
# online and offline; it is not a field in the source system.
con.execute("""
CREATE OR REPLACE VIEW grievances AS
SELECT
    ticket_no, created_on, resolved_on, status,
    mode,
    CASE mode_id
        WHEN 1 THEN 'Online' WHEN 5 THEN 'Online' WHEN 6 THEN 'Online'
        WHEN 7 THEN 'Online' WHEN 8 THEN 'Online' WHEN 9 THEN 'Online'
        WHEN 2 THEN 'Offline' WHEN 3 THEN 'Offline' WHEN 10 THEN 'Offline'
        WHEN 11 THEN 'Offline' WHEN 12 THEN 'Offline' WHEN 13 THEN 'Offline'
        WHEN 14 THEN 'Offline' WHEN 15 THEN 'Offline'
        ELSE 'Not stated'
    END AS channel,
    COALESCE(office, 'Not recorded') AS office,
    status = 'Disposed' AS is_resolved,
    status = 'Discard'  AS is_discarded,
    resolved_on IS NULL AS is_still_open,
    CASE WHEN resolved_on IS NOT NULL AND resolved_on >= created_on
         THEN CAST(resolved_on AS DATE) - CAST(created_on AS DATE) END AS days_to_close
FROM complaints
""")

con.execute("""
CREATE OR REPLACE VIEW resolved AS
SELECT * FROM grievances WHERE is_resolved AND days_to_close IS NOT NULL
""")
q("SELECT COUNT(*) AS n FROM grievances")

## 1. How many grievances, and when

Filing volume has grown steeply: the most recent complete year carries roughly
seven times the first complete year. Read the chart for the shape of that growth,
not for the final month, which is partial.

In [ ]:
by_year = facts("Grievances filed, April 2021 to July 2025", "grievances", "created_on",
                monthly_title="Grievances filed each month", save="01_volume_by_month")

In [ ]:
outcome = q("""
SELECT
    COUNT(*)                                     AS filed,
    COUNT(*) FILTER (WHERE is_resolved)          AS resolved,
    COUNT(*) FILTER (WHERE is_discarded)         AS closed_without_action,
    COUNT(*) FILTER (WHERE is_still_open)        AS still_open
FROM grievances
""")
r = outcome.row(0, named=True)
display(Markdown(
    f"Of the **{r['filed']:,}** grievances filed, **{r['resolved']:,}** "
    f"({100 * r['resolved'] / r['filed']:.0f}%) were resolved, "
    f"**{r['closed_without_action']:,}** ({100 * r['closed_without_action'] / r['filed']:.0f}%) "
    f"were closed without action, and **{r['still_open']:,}** "
    f"({100 * r['still_open'] / r['filed']:.0f}%) were still open when the data was taken."
))

fig, ax = plt.subplots(figsize=(9, 3.6))
d = year_col(q("""
    SELECT fy_label(created_on) AS year,
           COUNT(*) FILTER (WHERE is_resolved)   AS resolved,
           COUNT(*) FILTER (WHERE is_discarded)  AS closed_without_action,
           COUNT(*) FILTER (WHERE is_still_open) AS still_open
    FROM grievances GROUP BY 1 ORDER BY 1
"""))
bottom = [0] * d.height
for name, colour, label in [("resolved", MAROON, "Resolved"),
                            ("closed_without_action", BLUE, "Closed without action"),
                            ("still_open", ORANGE, "Still open")]:
    ax.bar(d["year"], d[name], bottom=bottom, color=colour, label=label,
           width=0.62, zorder=3, edgecolor="white", linewidth=2)
    bottom = [b + v for b, v in zip(bottom, d[name])]
thousands(ax)
ax.legend(loc="upper left")
ax.set_ylabel("Grievances")
finish(ax, "What happened to each year's grievances",
       subtitle="Recent years hold more open cases simply because less time has passed.",
       save="02_outcome_by_year")

## 2. Online or offline

Roughly seven grievances in ten arrive online, and the online share has been
rising. The single largest route is the website.

One judgement call sits behind these numbers: **Mobile** (43,710 grievances) is
counted as online, on the reading that it means the mobile app. If it means a
phone call, it belongs on the offline side. The mode-by-mode chart below is there
so that grouping can be checked rather than taken on trust.

In [ ]:
facts("Grievances filed online", "grievances", "created_on",
      where="channel = 'Online'",
      monthly_title="Online grievances each month", save="03_online_by_month")

share = year_col(q("""
    SELECT fy_label(created_on) AS year,
           ROUND(100.0 * COUNT(*) FILTER (WHERE channel = 'Online') / COUNT(*), 1) AS pct_online
    FROM grievances GROUP BY 1 ORDER BY 1
"""))
display(share.rename({"pct_online": "% filed online"}))

In [ ]:
m = q("""
    SELECT DATE_TRUNC('month', created_on) AS month,
           COUNT(*) FILTER (WHERE channel = 'Online')  AS online,
           COUNT(*) FILTER (WHERE channel = 'Offline') AS offline
    FROM grievances GROUP BY 1 ORDER BY 1
""")
fig, ax = plt.subplots(figsize=(9, 3.6))
ax.plot(m["month"], m["online"], color=MAROON, label="Online", zorder=3)
ax.plot(m["month"], m["offline"], color=BLUE, label="Offline", zorder=3)
ax.annotate("Online", (m["month"][-1], m["online"][-1]), xytext=(8, 0),
            textcoords="offset points", color=MAROON, fontsize=9,
            va="center", fontweight="bold")
ax.annotate("Offline", (m["month"][-1], m["offline"][-1]), xytext=(8, 0),
            textcoords="offset points", color=BLUE, fontsize=9, va="center")
ax.set_ylim(bottom=0)
ax.set_ylabel("Grievances")
thousands(ax)
ax.legend(loc="upper left")
finish(ax, "Online and offline filing, month by month",
       subtitle="Both routes grew; the gap between them widened from mid-2024.",
       save="04_online_offline_monthly")

In [ ]:
modes = q("""
    SELECT mode, channel, COUNT(*) AS n
    FROM grievances GROUP BY 1, 2 ORDER BY n DESC
""")
fig, ax = plt.subplots(figsize=(8.4, 5))
colours = [MAROON if c == "Online" else BLUE if c == "Offline" else ORANGE
           for c in modes["channel"]]
barh(ax, list(modes["mode"]), list(modes["n"]), color=colours)
thousands(ax, "x")
handles = [mpl.patches.Patch(color=MAROON, label="Online"),
           mpl.patches.Patch(color=BLUE, label="Offline"),
           mpl.patches.Patch(color=ORANGE, label="Not stated")]
ax.legend(handles=handles, loc="lower right")
ax.set_xlabel("Grievances filed")
finish(ax, "Every route people use to file",
       subtitle="Colour shows how each route was grouped into online or offline.",
       xgrid=True, ygrid=False, save="05_modes")

## 3. Which office receives the grievance

Half of everything goes to the District Collector. "Not recorded" is almost
entirely grievances arriving over social media, which carry no receiving office
in the record.

In [ ]:
facts("Grievances received by the District Collector", "grievances", "created_on",
      where="office = 'Collector'",
      monthly_title="Grievances to the District Collector, by month",
      save="06_collector_by_month")

offices = q("SELECT office, COUNT(*) AS n FROM grievances GROUP BY 1 ORDER BY n DESC")
fig, ax = plt.subplots(figsize=(8.4, 3.8))
barh(ax, list(offices["office"]), list(offices["n"]))
thousands(ax, "x")
ax.set_xlabel("Grievances received")
finish(ax, "Where grievances are received", xgrid=True, ygrid=False, save="07_offices")

In [ ]:
mix = year_col(q("""
    SELECT fy_label(created_on) AS year, office, COUNT(*) AS n
    FROM grievances GROUP BY 1, 2
"""))
wide = mix.pivot(on="office", index="year", values="n").fill_null(0).sort("year")
display(wide)

## 4. How long grievances take

Time to close is measured from the day a grievance is filed to the day it is
recorded as resolved.

**Two things to hold in mind before reading any number here.**

Grievances still open have no closing date, so they are left out entirely. That
makes every figure below a picture of the cases that *have* finished. Where a
group holds more unfinished cases than another, its figure will look better than
the full truth. The share left out is printed beside each comparison for exactly
that reason.

Grievances *closed without action* are counted separately from those *resolved*.
They close far faster, and mixing the two would drag every average down.

In [ ]:
facts("Typical days to close a resolved grievance", "resolved", "created_on",
      metric="PERCENTILE_CONT(0.5) WITHIN GROUP (ORDER BY days_to_close)",
      metric_name="Median days to close",
      monthly_title="Typical days to close, by month grievance was filed",
      monthly_ylabel="Median days",
      save="08_days_by_month", fmt="{:,.0f} days")

In [ ]:
cmp_ = q("""
    SELECT channel,
           COUNT(*) AS resolved_cases,
           PERCENTILE_CONT(0.5) WITHIN GROUP (ORDER BY days_to_close) AS median_days,
           PERCENTILE_CONT(0.9) WITHIN GROUP (ORDER BY days_to_close) AS slowest_tenth_days
    FROM resolved WHERE channel <> 'Not stated'
    GROUP BY 1 ORDER BY median_days DESC
""")
excl = q("""
    SELECT channel, ROUND(100.0 * COUNT(*) FILTER (WHERE is_still_open) / COUNT(*), 1) AS pct_still_open
    FROM grievances WHERE channel <> 'Not stated' GROUP BY 1
""")
display(cmp_.join(excl, on="channel").rename({
    "channel": "Channel", "resolved_cases": "Resolved cases",
    "median_days": "Typical days", "slowest_tenth_days": "Slowest tenth (days)",
    "pct_still_open": "% still open (excluded)"}))

byyear = year_col(q("""
    SELECT fy_label(created_on) AS year, channel,
           PERCENTILE_CONT(0.5) WITHIN GROUP (ORDER BY days_to_close) AS median_days
    FROM resolved WHERE channel <> 'Not stated' GROUP BY 1, 2 ORDER BY 1
"""))
fig, ax = plt.subplots(figsize=(9, 3.6))
w = 0.36
years = byyear["year"].unique(maintain_order=True).to_list()
for i, (ch, colour) in enumerate([("Online", MAROON), ("Offline", BLUE)]):
    sub = byyear.filter(pl.col("channel") == ch)
    vals = [sub.filter(pl.col("year") == y)["median_days"].to_list() or [0] for y in years]
    vals = [v[0] for v in vals]
    pos = [x + (i - 0.5) * w for x in range(len(years))]
    ax.bar(pos, vals, width=w, color=colour, label=ch, zorder=3)
    for xp, v in zip(pos, vals):
        ax.text(xp, v + 2, f"{v:,.0f}", ha="center", fontsize=8.5, color=INK)
ax.set_xticks(range(len(years)), years)
ax.set_ylabel("Median days to close")
ax.legend(loc="upper right")
finish(ax, "Typical days to close, online against offline",
       subtitle="Later years hold more unfinished cases, so their bars sit low.",
       save="09_days_channel_year")

In [ ]:
byoffice = q("""
    SELECT office,
           COUNT(*) AS resolved_cases,
           PERCENTILE_CONT(0.5) WITHIN GROUP (ORDER BY days_to_close) AS median_days
    FROM resolved GROUP BY 1 ORDER BY median_days DESC
""")
fig, ax = plt.subplots(figsize=(8.4, 3.8))
barh(ax, list(byoffice["office"]), list(byoffice["median_days"]), fmt="{:,.0f}")
ax.set_xlabel("Median days to close")
finish(ax, "Typical days to close, by receiving office",
       subtitle="Offices differ in what reaches them; this is not a ranking of effort.",
       xgrid=True, ygrid=False, save="10_days_by_office")
display(byoffice.rename({"office": "Office", "resolved_cases": "Resolved cases",
                         "median_days": "Typical days to close"}))

## 5. Where grievances wait

A grievance moves between offices before it closes. The record shows who acted
and when, so the gap between one recorded action and the next tells us how long a
case sat at each point.

The source system identifies each office by an internal code such as
`PDDRDA,Subarnapur` or `CEO,SHAS`. Those are not readable, so every office is
grouped here into the level of government it sits at, and into a plain-English
role. Individual office codes are not shown.

**A gap is not idle time.** It includes field enquiry, waiting periods required by
rule, and time spent waiting on the citizen for information. It is the time a case
sat between recorded steps, and nothing more precise than that.

In [ ]:
# Each recorded action, with the acting office translated out of its internal code.
con.execute("""
CREATE OR REPLACE VIEW acting_office AS
SELECT
    id, ticket_no, action_taken_date, action_status,
    CASE WHEN complaint_status_with_authority LIKE action_status || ' - %'
         THEN SUBSTR(complaint_status_with_authority, LENGTH(action_status) + 4)
    END AS code
FROM action_history
""")

con.execute("""
CREATE OR REPLACE VIEW acting_office_named AS
SELECT *,
  CASE
    WHEN code LIKE 'CM Grievance Cell%' OR code LIKE 'Chief Minister Office%'
      OR code LIKE 'Office of The Governor%' OR code LIKE 'Chief Secretary%'
      OR code LIKE 'Governor%'                                    THEN 'State apex'
    WHEN code LIKE 'Secretary %' OR code LIKE 'Director%' OR code LIKE 'Engineer-in-Chief%'
      OR code LIKE 'Directorate%' OR code LIKE 'Labour Commissioner%'
      OR code LIKE 'DGP%' OR code LIKE 'President, BSE%' OR code LIKE 'CEO,%'
      OR code LIKE 'Commissioner, %'                              THEN 'State department'
    WHEN code LIKE 'TPCODL%' OR code LIKE 'TPNODL%' OR code LIKE 'NESCO%'
      OR code LIKE 'TPSODL%' OR code LIKE 'TPWODL%' OR code LIKE 'WESCO%'
      OR code LIKE 'SOUTHCO%' OR code LIKE 'CESU%'                THEN 'Electricity utility'
    WHEN code LIKE 'Collector%' OR code LIKE 'Sub-Collector%' OR code LIKE 'SP,%'
      OR code LIKE 'DCP%' OR code LIKE 'DEO,%' OR code LIKE 'CDMO%' OR code LIKE 'DLO%'
      OR code LIKE 'DSWO%' OR code LIKE 'CDAO%' OR code LIKE 'DSSO%' OR code LIKE 'CSO%'
      OR code LIKE 'DWO%' OR code LIKE 'DFO%' OR code LIKE 'DRCS%' OR code LIKE 'DRDA%'
      OR code LIKE 'PDDRDA%' OR code LIKE 'DPO%' OR code LIKE 'DIC%'
      OR code LIKE 'R.W Division%' OR code LIKE 'R&B Division%'   THEN 'District office'
    WHEN code LIKE 'BDO%' OR code LIKE 'Tehsildar%' OR code LIKE 'IIC%' OR code LIKE 'SDPO%'
      OR code LIKE 'Executive Officer%' OR code LIKE 'Sarpanch%'
      OR code LIKE 'Sub-Divisional%' OR code LIKE 'ABDO%'         THEN 'Block or field office'
    ELSE 'Other office'
  END AS tier,
  CASE
    WHEN code LIKE 'BDO%' OR code LIKE 'ABDO%'          THEN 'Block Development Officer'
    WHEN code LIKE 'Tehsildar%'                          THEN 'Tahasildar (revenue)'
    WHEN code LIKE 'IIC%'                                THEN 'Police station'
    WHEN code LIKE 'SP,%' OR code LIKE 'SDPO%' OR code LIKE 'DCP%'
      OR code LIKE 'DGP%'                                THEN 'Police, district and above'
    WHEN code LIKE 'Sub-Collector%' OR code LIKE 'Sub-Divisional%'
                                                         THEN 'Sub-Collector'
    WHEN code LIKE 'Collector%'                          THEN 'District Collector'
    WHEN code LIKE 'PDDRDA%' OR code LIKE 'DRDA%'        THEN 'Rural development agency'
    WHEN code LIKE 'DEO,%'                               THEN 'District education officer'
    WHEN code LIKE 'CDMO%'                               THEN 'District medical officer'
    WHEN code LIKE 'Executive Officer%' OR code LIKE 'Commissioner, %'
                                                         THEN 'Urban local body'
    WHEN code LIKE 'Secretary %'                         THEN 'Department Secretary'
    WHEN code LIKE 'Director%' OR code LIKE 'Directorate%'
      OR code LIKE 'Engineer-in-Chief%'                  THEN 'State directorate'
    WHEN code LIKE 'CM Grievance Cell%' OR code LIKE 'Chief Minister Office%'
      OR code LIKE 'Chief Secretary%' OR code LIKE 'Office of The Governor%'
                                                         THEN 'Chief Minister''s and apex offices'
    WHEN code LIKE 'TPCODL%' OR code LIKE 'TPNODL%' OR code LIKE 'NESCO%'
      OR code LIKE 'TPSODL%' OR code LIKE 'TPWODL%' OR code LIKE 'WESCO%'
      OR code LIKE 'SOUTHCO%' OR code LIKE 'CESU%'       THEN 'Electricity utility'
    ELSE 'Other district or field office'
  END AS role
FROM acting_office
""")

# Order of events per grievance. Undated rows cannot be placed in a sequence and
# are dropped rather than guessed at.
con.execute("""
CREATE OR REPLACE VIEW steps_ordered AS
SELECT
    id, ticket_no, action_taken_date, action_status, tier, role,
    LAG(action_taken_date)  OVER w AS prev_date,
    LAG(id)                 OVER w AS prev_id,
    LAG(tier)               OVER w AS from_tier,
    LAG(role)               OVER w AS from_role,
    LAG(action_status)      OVER w AS from_status,
    LEAD(action_taken_date) OVER w AS next_date,
    ROW_NUMBER()            OVER w AS step_index
FROM acting_office_named
WHERE ticket_no IS NOT NULL AND action_taken_date IS NOT NULL
WINDOW w AS (PARTITION BY ticket_no ORDER BY action_taken_date, id)
""")

# Materialised, not a view: twelve later queries read it, and as a view the
# window over 6.55M rows would be recomputed by every one of them.
con.execute("""
CREATE OR REPLACE TABLE waits AS
SELECT
    ticket_no, step_index,
    from_tier, tier AS to_tier, from_role, from_status,
    CAST(action_taken_date AS DATE) - CAST(prev_date AS DATE) AS wait_days,
    next_date IS NULL AS is_last,
    (prev_id IS NOT NULL AND id < prev_id) AS is_out_of_order
FROM steps_ordered
WHERE prev_date IS NOT NULL
""")
None

In [ ]:
# Correctness gate, deliberately silent. `waits` re-derives the interval logic of
# the shipped `handoff` mart so it can carry the tier and role labels the mart does
# not have. If that re-derivation ever drifts from the mart, this raises and the
# section below must not be quoted.
install(con, "action_type", "handoff")
_mart = q("SELECT * FROM handoff_coverage_summary").row(0, named=True)
_mine = q("""
    SELECT COUNT(*) AS emitted_intervals,
           COUNT(*) FILTER (WHERE is_out_of_order) AS invalid_order_intervals,
           COUNT(*) FILTER (WHERE is_last) AS trailing_open_intervals,
           COUNT(DISTINCT ticket_no) AS tickets_with_intervals
    FROM waits
""").row(0, named=True)
assert all(_mart[k] == _mine[k] for k in _mine), (
    f"waits has drifted from handoff.sql: mart={_mart} notebook={_mine}"
)
None

In [ ]:
steps_per_case = q("""
    SELECT PERCENTILE_CONT(0.5) WITHIN GROUP (ORDER BY n) AS median_steps,
           AVG(n) AS mean_steps
    FROM (SELECT ticket_no, COUNT(*) + 1 AS n FROM waits WHERE NOT is_out_of_order GROUP BY 1)
""").row(0, named=True)
total_wait = q("SELECT SUM(wait_days) AS d FROM waits WHERE NOT is_out_of_order")["d"][0]
display(Markdown(
    f"A grievance passes through **{steps_per_case['median_steps']:.0f} recorded steps** "
    f"typically ({steps_per_case['mean_steps']:.1f} on average). Across all grievances the "
    f"record accounts for **{total_wait:,.0f} office-days** of waiting between steps."
))

In [ ]:
tier_wait = q("""
    SELECT from_tier AS tier,
           COUNT(*) AS handovers,
           PERCENTILE_CONT(0.5) WITHIN GROUP (ORDER BY wait_days) AS median_days,
           PERCENTILE_CONT(0.9) WITHIN GROUP (ORDER BY wait_days) AS slowest_tenth,
           SUM(wait_days) AS total_days
    FROM waits WHERE NOT is_out_of_order AND from_tier IS NOT NULL
    GROUP BY 1 ORDER BY median_days DESC
""")
# Plotted as share of all waiting, not the median. Most handovers are same-day
# administrative steps, so every tier's median sits at 1-4 days and the chart
# would say almost nothing. Where the time piles up is the decision-relevant
# quantity; the medians stay in the table below.
share = tier_wait.with_columns(
    (100 * pl.col("total_days") / pl.col("total_days").sum()).alias("pct")
).sort("pct", descending=True)
fig, ax = plt.subplots(figsize=(8.4, 3.4))
barh(ax, list(share["tier"]), list(share["pct"]), fmt="{:.0f}%")
ax.set_xlabel("Share of all waiting time in the record")
finish(ax, "Where the waiting piles up, by level of government",
       subtitle="Levels differ in what reaches them; this is not a ranking of effort.",
       xgrid=True, ygrid=False, save="11_wait_by_tier")

t = tier_wait.with_columns(
    (100 * pl.col("total_days") / pl.col("total_days").sum()).round(1).alias("share_of_all_waiting")
)
display(t.rename({"tier": "Level", "handovers": "Handovers", "median_days": "Typical days",
                  "slowest_tenth": "Slowest tenth (days)", "total_days": "Total days",
                  "share_of_all_waiting": "% of all waiting"}))

In [ ]:
role_wait = q("""
    SELECT from_role AS role,
           COUNT(*) AS handovers,
           PERCENTILE_CONT(0.5) WITHIN GROUP (ORDER BY wait_days) AS median_days,
           PERCENTILE_CONT(0.9) WITHIN GROUP (ORDER BY wait_days) AS slowest_tenth,
           SUM(wait_days) AS total_days
    FROM waits WHERE NOT is_out_of_order AND from_role IS NOT NULL
    GROUP BY 1 HAVING COUNT(*) >= 1000
    ORDER BY slowest_tenth DESC, role
""")
# The slowest tenth, not the median. Half of all handovers are same-day, so the
# median flattens every office type to a day or two; the long waits are the tail.
fig, ax = plt.subplots(figsize=(8.4, 5))
barh(ax, list(role_wait["role"]), list(role_wait["slowest_tenth"]), fmt="{:,.0f}")
ax.set_xlabel("Days waited, slowest tenth of cases")
finish(ax, "Where the long waits happen, by type of office",
       subtitle="Slowest tenth of handovers. Only office types with 1,000+ handovers.",
       xgrid=True, ygrid=False, save="12_wait_by_role")
display(role_wait.rename({"role": "Type of office", "handovers": "Handovers",
                          "median_days": "Typical days", "slowest_tenth": "Slowest tenth (days)",
                          "total_days": "Total days"}))

In [ ]:
stage = q("""
    SELECT step_index - 1 AS stage,
           COUNT(*) AS handovers,
           PERCENTILE_CONT(0.5) WITHIN GROUP (ORDER BY wait_days) AS median_days,
           PERCENTILE_CONT(0.9) WITHIN GROUP (ORDER BY wait_days) AS slowest_tenth
    FROM waits WHERE NOT is_out_of_order AND step_index <= 7
    GROUP BY 1 ORDER BY 1
""")
labels = {1: "1st to 2nd action", 2: "2nd to 3rd", 3: "3rd to 4th", 4: "4th to 5th",
          5: "5th to 6th", 6: "6th to 7th"}
s = stage.with_columns(
    pl.col("stage").replace_strict(labels, default="later").alias("stage_label")
)
fig, ax = plt.subplots(figsize=(9, 3.6))
ax.bar(s["stage_label"], s["median_days"], color=MAROON, width=0.6, zorder=3)
for i, v in enumerate(s["median_days"]):
    ax.text(i, v + 0.25, f"{v:,.0f}", ha="center", fontsize=9, color=INK)
ax.set_ylabel("Typical days waiting")
finish(ax, "Nearly all the waiting happens at one point",
       subtitle="The gap between a grievance's second and third recorded action.",
       save="13_wait_by_stage")
display(s.select(["stage_label", "handovers", "median_days", "slowest_tenth"]).rename({
    "stage_label": "Between", "handovers": "Handovers",
    "median_days": "Typical days", "slowest_tenth": "Slowest tenth (days)"}))

In [ ]:
chain = q("""
    SELECT LEAST(c.n, 9) AS steps, COUNT(*) AS cases,
           PERCENTILE_CONT(0.5) WITHIN GROUP (ORDER BY r.days_to_close) AS median_days
    FROM resolved r
    JOIN (SELECT ticket_no, COUNT(*) + 1 AS n FROM waits WHERE NOT is_out_of_order GROUP BY 1) c
      USING (ticket_no)
    WHERE c.n >= 3
    GROUP BY 1 ORDER BY 1
""")
fig, ax = plt.subplots(figsize=(9, 3.4))
ax.plot(chain["steps"], chain["median_days"], color=MAROON, marker="o", zorder=3)
for x, y in zip(chain["steps"], chain["median_days"]):
    ax.annotate(f"{y:,.0f}", (x, y), xytext=(0, 8), textcoords="offset points",
                ha="center", fontsize=8.5, color=INK)
ax.set_xlabel("Recorded steps a grievance passed through")
ax.set_ylabel("Median days to close")
ax.set_xticks(list(chain["steps"]),
              [str(s) if s < 9 else "9 or more" for s in chain["steps"]])
finish(ax, "More hands, more time",
       subtitle="Cases touched by more offices take longer, steeply so past six steps.",
       save="14_steps_vs_days")

## 6. Choosing districts for a field deep dive

This section is the point of the analysis. It sets out, for each of Odisha's 30
districts, the four things that bear on where a field visit would be most
informative: how much comes in, how long it takes to close, how much is still
sitting open, and at which level of government the waiting happens.

**Filing volume is not problem severity.** A district files more partly because
more people live there and partly because more of them know the system exists. A
low count is not evidence of fewer problems.

In [ ]:
con.execute("""
CREATE OR REPLACE VIEW district_grievances AS
SELECT g.*, c.district
FROM grievances g JOIN complaints c USING (ticket_no)
WHERE c.district IS NOT NULL
""")

facts("Grievances with a district recorded", "district_grievances", "created_on",
      monthly_title="Grievances filed each month, districts only",
      save="15_district_volume_by_month")

spread = year_col(q("""
    SELECT fy_label(created_on) AS year,
           COUNT(DISTINCT district) AS districts_filing,
           CAST(MEDIAN(n) AS BIGINT) AS median_district,
           CAST(MAX(n) AS BIGINT) AS busiest_district
    FROM (SELECT fy_label(created_on) AS y, created_on, district, COUNT(*) OVER
                 (PARTITION BY fy_label(created_on), district) AS n
          FROM district_grievances)
    GROUP BY 1 ORDER BY 1
"""))
display(spread.rename({"districts_filing": "Districts filing",
                       "median_district": "Median district",
                       "busiest_district": "Busiest district"}))

In [ ]:
# 30 districts is far past the point where colour can carry identity, so each
# gets its own small panel against a shared scale instead.
monthly = q("""
    SELECT district, DATE_TRUNC('month', created_on) AS month, COUNT(*) AS n
    FROM district_grievances GROUP BY 1, 2
""")
order = (monthly.group_by("district").agg(pl.col("n").sum().alias("t"))
         .sort("t", descending=True)["district"].to_list())
ymax = monthly["n"].max()

fig, axes = plt.subplots(6, 5, figsize=(11.5, 8.2), sharex=True, sharey=True)
for ax, name in zip(axes.ravel(), order):
    d = monthly.filter(pl.col("district") == name).sort("month")
    ax.plot(d["month"], d["n"], color=MAROON, linewidth=1.4)
    ax.fill_between(d["month"], d["n"], color=MAROON, alpha=0.12)
    ax.set_title(name, fontsize=9, pad=3, loc="left", color=INK)
    ax.set_ylim(0, ymax * 1.05)
    ax.grid(False)
    # One tick a year: the default put five overlapping date labels in a 2in panel.
    ax.xaxis.set_major_locator(mdates.YearLocator())
    ax.xaxis.set_major_formatter(mdates.DateFormatter("%Y"))
    ax.tick_params(labelsize=7)
    for s in ("top", "right", "left"):
        ax.spines[s].set_visible(False)
    ax.set_yticks([])
for ax in axes.ravel()[len(order):]:
    ax.set_visible(False)
fig.suptitle("Monthly filings, every district on the same scale",
             x=0.005, y=1.005, ha="left", fontsize=12, fontweight="bold", color=INK)
fig.text(0.005, 0.972, "Panels ordered by total volume. Shared vertical scale, so panel height is comparable.",
         fontsize=9.5, color=INK_SOFT)
fig.tight_layout(rect=[0, 0, 1, 0.955])
fig.savefig(FIGDIR / "16_district_small_multiples.png", bbox_inches="tight", facecolor="white")
plt.show()

In [ ]:
profile = q("""
WITH tier_mix AS (
    SELECT c.district,
           SUM(w.wait_days) AS all_wait,
           SUM(w.wait_days) FILTER (WHERE w.from_tier = 'Block or field office') AS block_wait
    FROM waits w JOIN complaints c USING (ticket_no)
    WHERE NOT w.is_out_of_order AND c.district IS NOT NULL
    GROUP BY 1
)
SELECT g.district AS "District",
       COUNT(*) AS "Grievances",
       ROUND(100.0 * COUNT(*) FILTER (WHERE g.channel = 'Online') / COUNT(*), 1) AS "% online",
       CAST(PERCENTILE_CONT(0.5) WITHIN GROUP (ORDER BY g.days_to_close)
            FILTER (WHERE g.is_resolved) AS BIGINT) AS "Typical days to close",
       ROUND(100.0 * COUNT(*) FILTER (WHERE g.is_still_open) / COUNT(*), 1) AS "% still open",
       ROUND(100.0 * MAX(t.block_wait) / MAX(t.all_wait), 1) AS "% of waiting at block level"
FROM district_grievances g JOIN tier_mix t ON t.district = g.district
GROUP BY 1 ORDER BY "Typical days to close" DESC
""")
display(profile)

In [ ]:
p = profile.to_pandas()
fig, ax = plt.subplots(figsize=(9.2, 6))
ax.scatter(p["Grievances"], p["Typical days to close"], s=46, color=MAROON,
           zorder=3, edgecolor="white", linewidth=1.2)
mx, my = p["Grievances"].median(), p["Typical days to close"].median()
ax.axvline(mx, color=RULE, linewidth=1, zorder=1)
ax.axhline(my, color=RULE, linewidth=1, zorder=1)
label_points(ax, p["Grievances"], p["Typical days to close"], p["District"])
# Top-right is the high-volume, slow quadrant. Sits at 0.90 to clear the
# top-most district label.
ax.text(0.995, 0.90, "High volume, slow:\nstrongest case for a visit", transform=ax.transAxes,
        ha="right", va="top", fontsize=8.5, color=INK_SOFT)
ax.text(0.005, 0.015, "Low volume, fast", transform=ax.transAxes,
        ha="left", va="bottom", fontsize=8.5, color=INK_SOFT)
thousands(ax, "x")
ax.set_xlabel("Grievances filed")
ax.set_ylabel("Typical days to close")
finish(ax, "Which districts combine volume with slow resolution",
       subtitle="Lines mark the median district on each axis. Labels are districts.",
       xgrid=True, save="17_district_scatter")

In [ ]:
top = profile.sort("Typical days to close", descending=True).head(12)
fig, ax = plt.subplots(figsize=(8.4, 4.6))
barh(ax, list(top["District"]), list(top["Typical days to close"]), fmt="{:,.0f}")
ax.set_xlabel("Typical days to close")
finish(ax, "The twelve slowest districts to close a grievance",
       subtitle="Read with the open-case share: a district can look fast by leaving cases open.",
       xgrid=True, ygrid=False, save="18_slowest_districts")

openest = profile.sort("% still open", descending=True).head(12)
fig, ax = plt.subplots(figsize=(8.4, 4.6))
barh(ax, list(openest["District"]), list(openest["% still open"]), color=BLUE, fmt="{:.1f}%")
ax.set_xlabel("% of grievances still open")
finish(ax, "The twelve districts carrying the most unfinished work",
       subtitle="These are the cases missing from every timing figure in this notebook.",
       xgrid=True, ygrid=False, save="19_most_open_districts")

### Reading this for site selection

Three things are worth pulling out of the table above.

**Slow and busy is not the same as backlogged.** A district can post a short
typical closing time while holding a large share of cases open, because only the
cases that finished are counted in the timing. Look at both columns together; a
district high on the open-case chart and low on the slow-list is closing its easy
cases and parking the rest.

**The block-level share tells you who to talk to.** Where most of a district's
waiting sits at block or field level, the useful conversation is with Block
Development Officers and Tahasildars. Where it sits at district or state level,
it is with the Collectorate or the department.

**Contrast pairs beat extremes.** Two districts of similar size and very different
closing times will teach more in a field visit than the two slowest, because size
is held roughly constant between them. The scatter is laid out to make those pairs
easy to spot: look for points at similar horizontal position and different height.

In [ ]:
con.close()

## What these numbers can and cannot say

**They describe what was recorded, not what was decided.** Every figure comes from
the log of actions officers entered. It shows what actually happened to each
grievance, not the route it was supposed to take.

**Waiting time is not idle time.** The gap between two recorded steps contains
field enquiry, waiting periods required by rule, and time spent waiting on the
citizen to respond. A long gap is not proof that anyone was slow.

**Nothing here is a comparison of effort between offices.** Offices and levels of
government receive very different kinds of case. A level that handles harder
grievances will show longer waits for that reason alone. These figures show where
time accumulates, not who is performing well.

**Unfinished grievances are left out of every timing figure.** They have no
closing date. Recent years therefore look faster than they will eventually prove
to be, because their slowest cases have not finished yet. The share left out is
printed beside each comparison.

**Closed without action is counted apart from resolved.** The two mean different
things and close on very different timescales.

**The online and offline grouping is ours.** The source system records fifteen
filing routes and does not label any of them online or offline. The route-by-route
chart in section 2 is there so the grouping can be checked.

**Office names are grouped, not quoted.** The source identifies each office by an
internal code. Those codes are grouped into levels of government and plain-English
roles; about 3% do not match any known pattern and appear as "Other office".

**No individual officer appears anywhere in this analysis.** The source system
holds a free-text name for whoever recorded each action. It is never read here.
Every figure is an aggregate, and no citizen's own words are used at any point.